# ETL & Data Preparation

The [EDA chapter](../01b-eda/exploratory-data-analysis.ipynb) closed with a
**data quality summary** — a short list of problems in `customers.csv`. This
chapter opens by addressing that list point by point:

1. **Missing values** in `age` and `income` → decide drop vs. impute.
2. **Outlier** income (~900k) → decide cap vs. remove.
3. **Inconsistent text** in `city` (`"berlin"`, `"Rome "`) → normalise.
4. **A duplicate-looking row** → deduplicate.
5. **Mild class imbalance** (~4:1) → handle for training.

We structure the work as **Extract → Transform → Load** (the "ETL" framing),
finishing with a reusable pipeline and a validation gate. The output is a clean,
model-ready dataset the [Model Evaluation](../01d-evaluation/cross-validation.ipynb)
and modelling chapters can consume directly.

## Extract

Read the messy dataset the EDA chapter profiled. `polars` reads CSV, JSON, and
Parquet with the same shape of API (see the
[I/O table in the appendix](../appendix/crate-reference.md)); CSV is our source
here, and the **Load** section below writes the cleaned result back out to
Parquet.

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "ndarray", "parquet", "strings"] }
use polars::prelude::*;

let raw = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("/book/data/customers.csv".into()))?
    .finish()?;
println!("raw shape = {:?}", raw.shape());
println!("{}", raw.head(Some(5)));

### The lazy API — a different mental model from pandas

`polars`' lazy API builds a **query plan** and only runs it at `.collect()`. The
optimiser then pushes filters and column projections down so it touches as
little data as possible — the idiom behind `scan_csv`/`scan_parquet` for building
pipelines without materialising intermediate results. Here we `.explain()` a
small plan to see that optimisation:

In [ ]:
let plan = raw.clone().lazy()
    .filter(col("churned").eq(lit(1)))
    .select([col("customer_id"), col("income"), col("tenure_months")]);

println!("optimised plan:\n{}", plan.explain(true)?);
let churned = plan.collect()?;
println!("\nchurned customers (first rows):\n{}", churned.head(Some(3)));

## Transform — cleaning

### Missing values (EDA problem 1)

`age` and `income` read as integers, and an integer column can't hold a float
mean, so we cast both to `Float64` first, then impute:

- `income` → **median** (skewed by the ~900k outlier, so the median is robust),
- `age` → **mean** (roughly symmetric, only one value missing).

This maps each choice back to what the EDA chapter's distribution analysis found,
rather than imputing arbitrarily.

In [ ]:
let imputed = raw.clone().lazy()
    .with_columns([
        col("age").cast(DataType::Float64),
        col("income").cast(DataType::Float64),
    ])
    .with_columns([
        col("age").fill_null(col("age").mean()),
        col("income").fill_null(col("income").median()),
    ])
    .collect()?;

println!("nulls after imputation:\n{}", imputed.null_count());

### Outliers (EDA problem 2)

The EDA chapter flagged the ~900k income with the IQR rule. We **winsorize** —
cap values above the upper fence `Q3 + 1.5·IQR` — rather than delete the row, so
we keep the customer while removing the distorting magnitude. The fence is
computed the same way the EDA chapter did (sorted quantiles).

In [ ]:
let fence: f64 = {
    let mut v: Vec<f64> = {
        let m = imputed.clone().lazy()
            .select([col("income")])
            .collect()?
            .to_ndarray::<Float64Type>(IndexOrder::C)?;
        (0..m.nrows()).map(|i| m[[i, 0]]).collect()
    };
    v.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let q = |p: f64| v[((v.len() as f64 - 1.0) * p).round() as usize];
    let (q1, q3) = (q(0.25), q(0.75));
    q3 + 1.5 * (q3 - q1)
};

let capped = imputed.clone().lazy()
    .with_columns([
        when(col("income").gt(lit(fence)))
            .then(lit(fence))
            .otherwise(col("income"))
            .alias("income"),
    ])
    .collect()?;

println!("upper fence = {:.0}", fence);
println!("max income before winsorising:\n{}",
    imputed.clone().lazy().select([col("income").max().alias("max_income")]).collect()?);
println!("max income after winsorising:\n{}",
    capped.clone().lazy().select([col("income").max().alias("max_income")]).collect()?);

### Inconsistent text & duplicates (EDA problems 3 & 4)

`city` has trailing spaces (`"Rome "`) and mixed case (`"berlin"`). We trim
whitespace and lower-case to a **canonical form** so the variants collapse. Then
we drop `customer_id` (an identifier, not a feature) and remove rows that are
identical on every remaining column — a record matching another on everything but
its ID is a likely double-entry (row `1034` duplicates `1002`).

In [ ]:
let cleaned = {
    let normed = capped.clone().lazy()
        .with_columns([
            col("city").str().strip_chars(lit(" ")).str().to_lowercase().alias("city"),
        ])
        .collect()?;
    // Drop the identifier, then dedup identical records.
    normed.drop("customer_id")?
        .unique_stable(None, UniqueKeepStrategy::First, None)?
};

println!("rows before dedup: {}  after: {}", capped.height(), cleaned.height());
println!("city value counts after normalisation:\n{}",
    cleaned.clone().lazy()
        .group_by([col("city")])
        .agg([len().alias("n")])
        .sort(["city"], Default::default())
        .collect()?);

## Transform — feature engineering

### Encoding categoricals

`city` is **nominal** (no inherent order) → **one-hot encode** it: one 0/1 column
per category. `tenure` is naturally **ordered**, so we bucket it into an
**ordinal** feature (short < medium < long) as a single integer. The contrast is
the point: one-hot for nominal, ordinal only when the categories genuinely have
an order.

In [ ]:
let cats = ["paris", "berlin", "rome", "madrid"];
let onehot: Vec<Expr> = cats.iter()
    .map(|c| col("city").eq(lit(*c)).cast(DataType::Int32).alias(format!("city_{}", c)))
    .collect();

let encoded = cleaned.clone().lazy()
    .with_columns(onehot)
    .with_columns([
        when(col("tenure_months").lt(lit(12))).then(lit(0))
            .when(col("tenure_months").lt(lit(36))).then(lit(1))
            .otherwise(lit(2))
            .alias("tenure_bucket"),
    ])
    .collect()?;

println!("{}", encoded.clone().lazy()
    .select([col("city"), col("city_paris"), col("city_berlin"),
             col("tenure_months"), col("tenure_bucket")])
    .collect()?
    .head(Some(6)));

### Scaling & derived features

**When scaling matters** ties the whole book together: distance-based models
(k-means / DBSCAN from the [Clustering chapter](../03-clustering/kmeans.ipynb))
and linear models care about feature scale; **tree models**
([Trees chapter](../04-trees/decision-trees.ipynb)) are scale-invariant, so this
step is model-dependent, not always required. We add both a **standardised**
(z-score) and a **min-max** scaled column, plus one **derived** ratio feature.

In [ ]:
let engineered = encoded.clone().lazy()
    .with_columns([
        ((col("income") - col("income").mean()) / col("income").std(1)).alias("income_z"),
        ((col("monthly_charge") - col("monthly_charge").min())
            / (col("monthly_charge").max() - col("monthly_charge").min())).alias("charge_minmax"),
        (col("monthly_charge") / col("income") * lit(1000.0)).alias("charge_per_1k_income"),
    ])
    .collect()?;

println!("{}", engineered.clone().lazy()
    .select([col("income"), col("income_z"),
             col("monthly_charge"), col("charge_minmax"), col("charge_per_1k_income")])
    .collect()?
    .head(Some(5)));

### Class imbalance (EDA problem 5)

Two standard responses to the ~4:1 imbalance:

- **Class weights** — weight each class by `n_total / (n_classes · n_class)` and
  pass those to a model that accepts them.
- **Resampling** — oversample the minority class.

Rust has **no mature SMOTE-style resampler** (a real gap vs. Python's
`imbalanced-learn`), so we do the simplest honest thing: naive random
oversampling by hand, duplicating the minority rows until the classes are roughly
balanced. This is the same "hand-rolled because no mature crate exists" pattern
the [Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter hits with
stratified K-fold.

In [ ]:
let counts = engineered.clone().lazy()
    .group_by([col("churned")])
    .agg([len().alias("n")])
    .sort(["churned"], Default::default())
    .collect()?;
println!("class counts:\n{}", counts);

// (a) class weights, by hand:
{
    let n_total = engineered.height() as f64;
    let ns: Vec<f64> = {
        let m = counts.clone().lazy()
            .select([col("n").cast(DataType::Float64)])
            .collect()?
            .to_ndarray::<Float64Type>(IndexOrder::C)?;
        (0..m.nrows()).map(|i| m[[i, 0]]).collect()
    };
    for (cls, n) in ns.iter().enumerate() {
        println!("class {} weight = {:.3}", cls, n_total / (2.0 * n));
    }
}

// (b) naive oversampling of the minority class (churned == 1):
let balanced = {
    let minority = engineered.clone().lazy().filter(col("churned").eq(lit(1))).collect()?;
    let majority_n = engineered.height() - minority.height();
    let copies = (majority_n / minority.height().max(1)).max(1);
    let mut out = engineered.clone();
    for _ in 1..copies {
        out.vstack_mut(&minority)?;
    }
    out
};
println!("balanced class counts:\n{}", balanced.clone().lazy()
    .group_by([col("churned")]).agg([len().alias("n")])
    .sort(["churned"], Default::default()).collect()?);

## Load

Two things happen at Load: **persist** the cleaned+engineered frame to a portable
Parquet artifact (which later chapters could load directly), and **convert** the
model-ready columns into the `ndarray` matrix form that `linfa`/`smartcore`
expect. That `X` / `y` split is exactly what the
[Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter consumes via
`Dataset::new(x, y)` — making this conversion explicit here means every later
chapter can reuse it.

In [ ]:
use std::fs::File;

// Persist to Parquet, then read back to confirm the round-trip.
{
    let mut d = engineered.clone();
    let f = File::create("/tmp/customers_clean.parquet")?;
    ParquetWriter::new(f).finish(&mut d)?;
}
let reloaded = ParquetReader::new(File::open("/tmp/customers_clean.parquet")?).finish()?;
println!("wrote & reloaded parquet: {:?}", reloaded.shape());

// Convert the model-ready columns into ndarray form. `to_ndarray` (from polars'
// `ndarray` feature) returns an Array2<f64>; we keep the arrays inside a block so
// evcxr needn't name the ndarray type across cells (see the crate-reference
// appendix). This X / y pair is what `Dataset::new(x, y)` consumes downstream.
let feature_cols = [
    "income_z", "charge_minmax", "charge_per_1k_income", "tenure_bucket",
    "city_paris", "city_berlin", "city_rome", "city_madrid",
];
{
    let x = engineered.clone().lazy()
        .select(feature_cols.iter().map(|c| col(*c).cast(DataType::Float64)).collect::<Vec<_>>())
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    let y = engineered.clone().lazy()
        .select([col("churned").cast(DataType::Float64)])
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    println!("X (features) shape = {:?}", x.shape());
    println!("y (target)   shape = {:?}  -> a single label column", y.shape());
}

## A reusable pipeline

The steps above, packaged into one function you can call on any new batch of raw
data. This is precisely what `automl`'s `PreprocessingPipeline` does for you
automatically (see the AutoML chapter) — here you see the manual version first,
so the automated one later isn't magic.

In [ ]:
fn prepare(raw: &DataFrame, fence: f64) -> PolarsResult<DataFrame> {
    raw.clone().lazy()
        .with_columns([
            col("age").cast(DataType::Float64),
            col("income").cast(DataType::Float64),
        ])
        .with_columns([
            col("age").fill_null(col("age").mean()),
            col("income").fill_null(col("income").median()),
        ])
        .with_columns([
            when(col("income").gt(lit(fence))).then(lit(fence)).otherwise(col("income")).alias("income"),
            col("city").str().strip_chars(lit(" ")).str().to_lowercase().alias("city"),
        ])
        .with_columns([
            ((col("income") - col("income").mean()) / col("income").std(1)).alias("income_z"),
        ])
        .collect()?
        .drop("customer_id")?
        .unique_stable(None, UniqueKeepStrategy::First, None)
}

let prepared = prepare(&raw, fence)?;
println!("pipeline output shape: {:?}", prepared.shape());
println!("{}", prepared.clone().lazy()
    .select([col("city"), col("income"), col("income_z")])
    .collect()?
    .head(Some(5)));

## Data validation before training

A final **gate** to run before any train/test split (the
[Model Evaluation](../01d-evaluation/cross-validation.ipynb) chapter): confirm the
expected schema is present, no nulls survived, and values sit in sane ranges. It
returns `Err` on the first violation, so a broken dataset never silently reaches
training.

In [ ]:
fn validate(df: &DataFrame) -> Result<(), String> {
    // Schema check: required columns present.
    for c in ["age", "income", "city", "churned"] {
        if df.column(c).is_err() {
            return Err(format!("missing expected column: {}", c));
        }
    }
    // No nulls survived.
    let total_nulls: usize = df.get_columns().iter().map(|c| c.null_count()).sum();
    if total_nulls > 0 {
        return Err(format!("{} null value(s) remain", total_nulls));
    }
    // Range check: plausible ages.
    let bad_age = df.clone().lazy()
        .filter(col("age").lt(lit(0.0)).or(col("age").gt(lit(120.0))))
        .collect().map_err(|e| e.to_string())?
        .height();
    if bad_age > 0 {
        return Err(format!("{} row(s) with implausible age", bad_age));
    }
    // Target is binary.
    let bad_target = df.clone().lazy()
        .filter(col("churned").neq(lit(0)).and(col("churned").neq(lit(1))))
        .collect().map_err(|e| e.to_string())?
        .height();
    if bad_target > 0 {
        return Err(format!("{} row(s) with target not in {{0, 1}}", bad_target));
    }
    Ok(())
}

match validate(&prepared) {
    Ok(())  => println!("validation passed — dataset is safe to train on"),
    Err(e)  => println!("validation FAILED: {}", e),
}

## Recap

We turned the EDA chapter's five-point problem list into a clean, validated,
model-ready dataset — imputed, winsorised, de-duplicated, encoded, scaled, and
balanced — then packaged it into a reusable `prepare()` function and a validation
gate. The Parquet artifact and the `X`/`y` `ndarray` split are the hand-off to the
rest of the book.

Next: [Model Evaluation & Cross-Validation](../01d-evaluation/cross-validation.ipynb),
which takes an `X`/`y` like this one and shows how to trust a model's reported
performance before believing it.